# 00 Smoke Test - Phase 0 환경 검증

**목적**: 모터-감속기 MiniROCKET 파이프라인 진입 전, 의존성/시드/로깅이 정상 동작하는지 확인.

검증 항목:
1. sktime / scikit-learn / numpy 버전이 핀과 일치 (`src/requirements.txt`)
2. `MiniRocket` import 경로 (`sktime.transformations.panel.rocket`) 유효
3. `set_global_seed(42)` 2회 호출 시 `np.random.rand()` 동일값 (NFR-03)
4. `MiniRocket().fit_transform(np.random.randn(2,1,10000))` 결과 shape ~ (2, ~9996)
5. 로거 INFO/WARN 한 줄씩 출력 및 파일 기록


## (Colab 전용) 의존성 설치
로컬에서 이미 `pip install -r src/requirements.txt` 를 마쳤다면 다음 셀은 스킵해도 된다.

In [1]:
# !pip install -q -r ../src/requirements.txt

## 1. import 및 버전 확인

In [2]:
import sys
import os

# notebooks/ 에서 실행해도 src/ 패키지를 import 할 수 있도록 프로젝트 루트를 sys.path 에 추가
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import sklearn
import sktime
from sktime.transformations.panel.rocket import MiniRocket

print(f"python      : {sys.version.split()[0]}")
print(f"numpy       : {np.__version__}")
print(f"sklearn     : {sklearn.__version__}")
print(f"sktime      : {sktime.__version__}")
print(f"MiniRocket  : {MiniRocket}")

python      : 3.11.0
numpy       : 1.26.4
sklearn     : 1.7.2
sktime      : 0.40.1
MiniRocket  : <class 'sktime.transformations.panel.rocket._minirocket.MiniRocket'>


## 2. 시드 재현성 검증 (V0-3)

In [3]:
from src.common.seed import set_global_seed

set_global_seed(42)
first = np.random.rand(5)

set_global_seed(42)
second = np.random.rand(5)

print("first :", first)
print("second:", second)
assert np.array_equal(first, second), "set_global_seed 재현성 실패 (NFR-03 위반)"
print("PASS: set_global_seed 두 번 호출 후 np.random.rand() 동일")

first : [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
second: [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
PASS: set_global_seed 두 번 호출 후 np.random.rand() 동일


## 3. MiniRocket smoke test (V0-2)

In [4]:
set_global_seed(42)
X = np.random.randn(2, 1, 2000).astype(np.float32)
transformer = MiniRocket(random_state=42)
Z = transformer.fit_transform(X)

print(f"input  shape: {X.shape}")
print(f"output shape: {Z.shape}")
# MiniRocket 기본 num_kernels=10_000, **시퀀스 길이와 무관하게** 실제 출력 피처 수는 약 9996 근방
# (윈도우 길이 PLAN 변경 10000 -> 2000 반영: 입력 길이만 단축, 피처 차원은 그대로)
assert Z.shape[0] == 2
assert 9000 <= Z.shape[1] <= 11000, f"unexpected feature dim: {Z.shape[1]}"
print("PASS: MiniRocket 변환 shape 정상")

input  shape: (2, 1, 2000)
output shape: (2, 9996)
PASS: MiniRocket 변환 shape 정상


## 4. 로거 동작 확인

In [5]:
from src.common.logger import get_logger

log_dir = os.path.join(PROJECT_ROOT, "logs")
logger = get_logger("smoke", log_dir=log_dir)

logger.info("INFO: smoke test 콘솔 출력 확인")
logger.warning("WARN: 파일(logs/smoke.log)에도 기록되는지 확인")

log_file = os.path.join(log_dir, "smoke.log")
assert os.path.exists(log_file), "로그 파일이 생성되지 않음"
with open(log_file, "r", encoding="utf-8") as f:
    tail = f.readlines()[-3:]
print("--- logs/smoke.log (tail) ---")
for line in tail:
    print(line.rstrip())
print("PASS: 로거 INFO/WARN 동작 확인")

2026-05-16 22:29:23,502 [INFO] smoke: INFO: smoke test 콘솔 출력 확인


2026-05-16 22:29:23,503 [WARNING] smoke: WARN: 파일(logs/smoke.log)에도 기록되는지 확인


--- logs/smoke.log (tail) ---
2026-05-16 20:58:46,710 [WARNING] smoke: WARN line for smoke test
2026-05-16 21:17:42,055 [WARNING] smoke: WARN: 파일(logs/smoke.log)에도 기록되는지 확인
2026-05-16 22:29:23,503 [WARNING] smoke: WARN: 파일(logs/smoke.log)에도 기록되는지 확인
PASS: 로거 INFO/WARN 동작 확인


---
모든 셀이 통과하면 **Phase 0 완료**. Phase 1 (데이터 인덱싱 + CSVLoader) 로 진행.